# KeyMonkey — Piano Note Prediction: Dataset Preprocessing

Converts the raw MAESTRO v3 dataset into cached `.npz` piano roll files ready for model training.

### What this notebook does
1. Installs dependencies and mounts Google Drive
2. Creates the cache and checkpoint folders if they don't exist
3. Reads the MAESTRO CSV to get the train/validation/test split assignments
4. Converts every MIDI file into a binary piano roll matrix at 32 Hz and saves it as a `.npz` file
5. Writes `train_index.txt`, `validation_index.txt`, `test_index.txt` so other notebooks know which files belong to which split
6. Computes `pos_weight` from training data only
7. Builds PyTorch DataLoaders and verifies a batch loads correctly

### Tensor structure
- **Piano roll:** shape `(T_frames, 88_keys)` — one row per 1/32-second time slice, 88 binary columns (A0–C8)
- **Input `x`:** shape `(B, 511, 88)` — frames 0 → 510
- **Target `y`:** shape `(B, 511, 88)` — frames 1 → 511, shifted forward one frame for autoregressive prediction


## Step 1 - Install Dependencies and Mount Drive

In [ ]:
!pip install pretty_midi -q

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import pretty_midi
import torch
import torch.nn as nn
import random as _random
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

DRIVE_ROOT = Path('/content/drive/My Drive/Key Monkey')
DATA_ROOT  = DRIVE_ROOT / 'maestro-v3.0.0'
CACHE_DIR  = DRIVE_ROOT / 'maestro_cache'
CKPT_DIR   = DRIVE_ROOT / 'maestro_checkpoints'

assert DATA_ROOT.exists(), f'MAESTRO dataset not found at: {DATA_ROOT}'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset:     {DATA_ROOT}')
print(f'Cache:       {CACHE_DIR}')
print(f'Checkpoints: {CKPT_DIR}')
print('Folders ready.')


Mounted at /content/drive
Dataset:     /content/drive/My Drive/Key Monkey/maestro-v3.0.0
Cache:       /content/drive/My Drive/Key Monkey/maestro_cache
Checkpoints: /content/drive/My Drive/Key Monkey/maestro_checkpoints
Folders ready.


## Step 2 - Global Constants

In [ ]:
PIANO_MIN  = 21
N_PITCHES  = 88
FRAME_RATE = 32
MAX_FRAMES = 512
BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Frame rate : {FRAME_RATE} Hz  →  {MAX_FRAMES} frames = {MAX_FRAMES / FRAME_RATE:.1f} seconds')


Device: cpu
Frame rate : 32 Hz  →  512 frames = 16.0 seconds


## Step 3 - Build the Piano Roll Cache

This is the main preprocessing step. For every MIDI file in the MAESTRO dataset:

1. Load the file with `pretty_midi`
2. Extract a binary piano roll at 32 frames per second — each cell is `1.0` if that key is sounding at that moment, `0.0` otherwise
3. Clip to the 88 standard piano keys (MIDI 21–108)
4. Save as a compressed `.npz` file in `maestro_cache/`

Files already processed are skipped.

The MAESTRO CSV contains a `split` column (`train`, `validation`, `test`) which is used directly
— no re-splitting is done here. This preserves the original composition-level split that prevents
the same pianist/performance appearing in multiple partitions.


In [ ]:
def midi_to_piano_roll(midi_path, frame_rate=FRAME_RATE, piano_min=PIANO_MIN, n_pitches=N_PITCHES):
    """
    Convert a MIDI file to a binary piano roll matrix.

    Returns:
        np.ndarray of shape (T_frames, 88) with dtype float32.
        Each cell is 1.0 if the key is active at that frame, 0.0 otherwise.
        Returns None if the file cannot be parsed.
    """
    try:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
    except Exception as e:
        print(f'  Skipping {midi_path.name}: {e}')
        return None

    roll_full = pm.get_piano_roll(fs=frame_rate)

    if roll_full is None or roll_full.shape[1] == 0:
        return None

    roll_full = roll_full.T.astype(np.float32)

    piano_max = piano_min + n_pitches
    roll = roll_full[:, piano_min:piano_max]

    roll = (roll > 0).astype(np.float32)

    return roll


def build_cache(data_root, cache_dir, frame_rate=FRAME_RATE):
    """
    Process all MIDI files listed in the MAESTRO CSV and save piano rolls to cache_dir.
    Writes split index files: train_index.txt, validation_index.txt, test_index.txt.
    """
    csv_candidates = list(data_root.glob('*.csv'))
    assert len(csv_candidates) > 0, f'No CSV found in {data_root}'
    csv_path = csv_candidates[0]
    print(f'Reading metadata from: {csv_path.name}')

    df = pd.read_csv(str(csv_path))
    print(f'Total files in dataset: {len(df)}')
    print(df['split'].value_counts().to_string())
    print()

    split_files = {'train': [], 'validation': [], 'test': []}

    skipped  = 0
    saved    = 0
    existing = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing MIDI files'):
        midi_rel  = row['midi_filename']
        split     = row['split']
        midi_path = data_root / midi_rel

        if not midi_path.exists():
            print(f'  Missing: {midi_path}')
            skipped += 1
            continue

        safe_name = midi_rel.replace('/', '__').replace('\\', '__')
        npz_name  = safe_name.replace('.midi', '.npz').replace('.mid', '.npz')
        npz_path  = cache_dir / npz_name

        if split in split_files:
            split_files[split].append(npz_name)

        if npz_path.exists():
            existing += 1
            continue

        roll = midi_to_piano_roll(midi_path, frame_rate=frame_rate)
        if roll is None or roll.shape[0] < 32:
            skipped += 1
            continue

        np.savez_compressed(str(npz_path), roll=roll)
        saved += 1

    for split_name, files in split_files.items():
        index_path = cache_dir / f'{split_name}_index.txt'
        valid_files = [f for f in files if (cache_dir / f).exists()]
        with open(str(index_path), 'w') as fh:
            fh.write('\n'.join(valid_files))
        print(f'{split_name:12s}: {len(valid_files)} files  →  {index_path.name}')

    print(f'\nSaved:    {saved} new files')
    print(f'Existing: {existing} already cached')
    print(f'Skipped:  {skipped} (missing or too short)')
    print('Cache build complete.')


build_cache(DATA_ROOT, CACHE_DIR)


Reading metadata from: maestro-v3.0.0.csv
Total files in dataset: 1276
split
train         962
test          177
validation    137



Processing MIDI files:   0%|          | 0/1276 [00:00<?, ?it/s]

train       : 962 files  →  train_index.txt
validation  : 137 files  →  validation_index.txt
test        : 177 files  →  test_index.txt

Saved:    1276 new files
Existing: 0 already cached
Skipped:  0 (missing or too short)
Cache build complete.


## Step 4 — Verify the Cache

In [ ]:
for split in ['train', 'validation', 'test']:
    index_path = CACHE_DIR / f'{split}_index.txt'
    assert index_path.exists(), f'Missing index: {index_path}'
    with open(str(index_path)) as f:
        files = [l.strip() for l in f if l.strip()]
    print(f'{split:12s}: {len(files)} files')

sample_file = CACHE_DIR / files[0]
roll = np.load(str(sample_file))['roll']
print(f'\nSample roll shape: {roll.shape}  (frames x pitches)')
print(f'Duration at 32 Hz: {roll.shape[0] / FRAME_RATE:.1f} seconds')
print(f'Active cells:      {roll.mean()*100:.2f}%')


train       : 962 files
validation  : 137 files
test        : 177 files

Sample roll shape: (5247, 88)  (frames x pitches)
Duration at 32 Hz: 164.0 seconds
Active cells:      7.83%


## Step 5 — Compute Positive Weight

Scans a random sample of training files to measure the silence-to-note ratio.
This value becomes `pos_weight` in `BCEWithLogitsLoss`, preventing the model from
collapsing to predicting all-silence.

Only training data is used here — no leakage from validation or test.


In [ ]:
def compute_pos_weight(cache_dir, split='train', n_files=300, seed=42):
    rng = np.random.default_rng(seed)
    index_path = Path(cache_dir) / f'{split}_index.txt'
    with open(index_path) as f:
        files = [Path(cache_dir) / l.strip() for l in f if l.strip()]

    rng.shuffle(files)
    files = files[:n_files]

    total_cells = active_cells = 0

    for fp in files:
        try:
            roll = np.load(str(fp))['roll']
            total_cells  += roll.size
            active_cells += (roll > 0.5).sum()
        except Exception:
            continue

    silent_cells = total_cells - active_cells
    ratio = silent_cells / max(active_cells, 1)

    print(f'  Split:         {split}')
    print(f'  Files sampled: {n_files}')
    print(f'  Active cells:  {active_cells:,}')
    print(f'  Silent cells:  {silent_cells:,}')
    print(f'  Ratio:         {ratio:.1f} : 1')
    return float(ratio)

computed_pos_weight = compute_pos_weight(CACHE_DIR)
print(f'\npos_weight = {computed_pos_weight:.2f}')


  Split:         train
  Files sampled: 300
  Active cells:  27,494,062
  Silent cells:  468,310,962
  Ratio:         17.0 : 1

pos_weight = 17.03


## Step 6 — Build DataLoaders and Run a Sanity Check

In [ ]:
class MaestroDataset(Dataset):
    def __init__(self, cache_dir, split, max_frames=MAX_FRAMES):
        index_path = Path(cache_dir) / f'{split}_index.txt'
        with open(index_path) as f:
            self.files = [Path(cache_dir) / l.strip() for l in f if l.strip()]
        self.max_frames = max_frames

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        roll = np.load(str(self.files[idx]))['roll'].astype(np.float32)
        T    = roll.shape[0]

        if T >= self.max_frames:
            start = _random.randint(0, T - self.max_frames)
            clip  = roll[start : start + self.max_frames]
        else:
            pad  = np.zeros((self.max_frames - T, N_PITCHES), dtype=np.float32)
            clip = np.vstack([roll, pad])

        x = torch.from_numpy(clip[:-1])
        y = torch.from_numpy(clip[1:])
        return x, y


train_ds = MaestroDataset(CACHE_DIR, 'train')
val_ds   = MaestroDataset(CACHE_DIR, 'validation')
test_ds  = MaestroDataset(CACHE_DIR, 'test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

x_batch, y_batch = next(iter(train_loader))
print(f'\nBatch x shape: {tuple(x_batch.shape)}  (batch, frames, pitches)')
print(f'Batch y shape: {tuple(y_batch.shape)}')
print(f'Active notes in batch: {y_batch.mean()*100:.2f}%')
print('\nPreprocessing complete. You can now run the model notebooks.')


Train: 962 | Val: 137 | Test: 177


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Batch x shape: (32, 511, 88)  (batch, frames, pitches)
Batch y shape: (32, 511, 88)
Active notes in batch: 5.57%

Preprocessing complete. You can now run the model notebooks.
